# Projeto Sprint 8 — Recomendação de planos Megaline

**Objetivo:** desenvolver um modelo de classificação que analise o comportamento dos clientes e recomende um dos planos mais recentes da Megaline (Smart ou Ultra), com acurácia mínima de 0,75.

## 1. Abrir e examinar o arquivo de dados

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

In [26]:
df = pd.read_csv('data/users_behavior.csv')

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [17]:
df.describe()

,calls,minutes,messages,mb_used,is_ultra
count,3214.000000,3214.000000,3214.000000,3214.000000,3214.000000
mean,63.038892,438.208787,38.281269,17207.673836,0.306472
std,33.236368,234.569872,36.148326,7570.968246,0.461100
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,40.000000,274.575000,9.000000,12491.902500,0.000000
50%,62.000000,430.600000,30.000000,16943.235000,0.000000
75%,82.000000,571.927500,57.000000,21424.700000,1.000000
max,244.000000,1632.060000,224.000000,49745.730000,1.000000


In [18]:
# Verificando valores ausentes e duplicados
print('Valores ausentes por coluna:')
print(df.isna().sum())
print()
print('Linhas duplicadas:', df.duplicated().sum())
print()
print('Distribuição da variável alvo (is_ultra):')
print(df['is_ultra'].value_counts(normalize=True))

Valores ausentes por coluna:
calls       0
minutes     0
messages    0
mb_used     0
is_ultra    0
dtype: int64

Linhas duplicadas: 0

Distribuição da variável alvo (is_ultra):
is_ultra
0    0.693528
1    0.306472
Name: proportion, dtype: float64


**Observações sobre os dados:**
- O dataset tem 3214 linhas e 5 colunas, sem valores ausentes e sem linhas duplicadas.
- As colunas `calls`, `minutes`, `messages` e `mb_used` são as características (features); `is_ultra` é o objetivo (target), já que o pré-processamento foi feito no projeto anterior.
- A classe é desbalanceada: cerca de 69,4% dos clientes usam o plano Smart (0) e 30,6% usam o Ultra (1). Isso é importante ter em mente ao avaliar a acurácia dos modelos: um modelo "bobo" que sempre prevê a classe majoritária já acertaria ~69% das vezes.

## 2. Dividir os dados em treinamento, validação e teste

Como não recebemos um conjunto de teste separado, vamos dividir os dados originais em três partes:
- **60%** para treinamento
- **20%** para validação (usado para escolher hiperparâmetros)
- **20%** para teste (usado só no final, para a avaliação final do modelo escolhido)

In [19]:
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

# Primeiro separamos 60% para treino e 40% para o restante (validação + teste)
features_train, features_valid_test, target_train, target_valid_test = train_test_split(
    features, target, test_size=0.4, random_state=12345
)

# Depois dividimos os 40% restantes ao meio: 20% validação, 20% teste
features_valid, features_test, target_valid, target_test = train_test_split(
    features_valid_test, target_valid_test, test_size=0.5, random_state=12345
)

print('Treinamento:', features_train.shape)
print('Validação:  ', features_valid.shape)
print('Teste:      ', features_test.shape)

Treinamento: (1928, 4)
Validação:   (643, 4)
Teste:       (643, 4)


**Sobre a escolha dos tamanhos:** usamos a proporção comum 3:1:1 (60% / 20% / 20%). O conjunto de treinamento precisa ser o maior, para que o modelo tenha dados suficientes para aprender os padrões. Os conjuntos de validação e teste ficam do mesmo tamanho: a validação é usada repetidamente para comparar hiperparâmetros, enquanto o teste é reservado e usado uma única vez, no final, para simular dados totalmente novos.

## 3. Investigando a qualidade de diferentes modelos

### 3.1 Árvore de decisão (Decision Tree)

In [20]:
best_tree_model = None
best_tree_result = 0
best_tree_depth = 0

for depth in range(1, 11):
    model = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model.fit(features_train, target_train)
    predictions_valid = model.predict(features_valid)
    result = accuracy_score(target_valid, predictions_valid)

    print(f'max_depth = {depth}: acurácia (validação) = {result:.4f}')

    if result > best_tree_result:
        best_tree_model = model
        best_tree_result = result
        best_tree_depth = depth

print()
print(f'Melhor árvore: max_depth = {best_tree_depth}, acurácia = {best_tree_result:.4f}')

max_depth = 1: acurácia (validação) = 0.7543
max_depth = 2: acurácia (validação) = 0.7823
max_depth = 3: acurácia (validação) = 0.7854
max_depth = 4: acurácia (validação) = 0.7792
max_depth = 5: acurácia (validação) = 0.7792


max_depth = 6: acurácia (validação) = 0.7838
max_depth = 7: acurácia (validação) = 0.7823
max_depth = 8: acurácia (validação) = 0.7792
max_depth = 9: acurácia (validação) = 0.7823
max_depth = 10: acurácia (validação) = 0.7745

Melhor árvore: max_depth = 3, acurácia = 0.7854


### 3.2 Floresta aleatória (Random Forest)

In [21]:
best_forest_model = None
best_forest_result = 0
best_forest_params = None

for est in range(10, 101, 10):
    for depth in range(1, 11):
        model = RandomForestClassifier(random_state=12345, n_estimators=est, max_depth=depth)
        model.fit(features_train, target_train)
        predictions_valid = model.predict(features_valid)
        result = accuracy_score(target_valid, predictions_valid)

        if result > best_forest_result:
            best_forest_model = model
            best_forest_result = result
            best_forest_params = (est, depth)

print(f'Melhor floresta: n_estimators = {best_forest_params[0]}, '
      f'max_depth = {best_forest_params[1]}, acurácia = {best_forest_result:.4f}')

Melhor floresta: n_estimators = 40, max_depth = 8, acurácia = 0.8087


### 3.3 Regressão logística (Logistic Regression)

In [22]:
logistic_model = LogisticRegression(random_state=12345, solver='liblinear')
logistic_model.fit(features_train, target_train)
predictions_valid = logistic_model.predict(features_valid)
logistic_result = accuracy_score(target_valid, predictions_valid)

print(f'Regressão logística: acurácia (validação) = {logistic_result:.4f}')

Regressão logística: acurácia (validação) = 0.7574


### 3.4 Comparando os modelos

In [23]:
results = pd.DataFrame({
    'Modelo': ['Árvore de decisão', 'Floresta aleatória', 'Regressão logística'],
    'Melhores hiperparâmetros': [
        f'max_depth={best_tree_depth}',
        f'n_estimators={best_forest_params[0]}, max_depth={best_forest_params[1]}',
        'solver=liblinear'
    ],
    'Acurácia (validação)': [best_tree_result, best_forest_result, logistic_result]
})

results.sort_values('Acurácia (validação)', ascending=False)

,Modelo,Melhores hiperparâmetros,Acurácia (validação)
1,Floresta aleatória,"n_estimators=40, max_depth=8",0.808709
0,Árvore de decisão,max_depth=3,0.785381
2,Regressão logística,solver=liblinear,0.757387


**Resultados do estudo:**
- A **árvore de decisão** teve seu melhor desempenho com uma profundidade relativamente baixa; profundidades maiores tendem a superajustar (overfitting) aos dados de treinamento, reduzindo a acurácia na validação.
- A **floresta aleatória** (combinação de várias árvores) obteve a melhor acurácia entre os três modelos, como esperado, florestas aleatórias tendem a generalizar melhor do que uma única árvore, pois combinam os votos de várias árvores treinadas com subconjuntos diferentes dos dados.
- A **regressão logística** teve o desempenho mais baixo dos três, o que é comum quando as fronteiras entre as classes não são bem representadas por uma combinação linear das características.
- Escolhemos a **floresta aleatória com os melhores hiperparâmetros encontrados** como modelo final, por ter obtido a maior acurácia no conjunto de validação.

## 4. Verificando a qualidade do modelo com o conjunto de teste

In [24]:
final_model = best_forest_model

test_predictions = final_model.predict(features_test)
test_accuracy = accuracy_score(target_test, test_predictions)

print(f'Acurácia no conjunto de teste: {test_accuracy:.4f}')

if test_accuracy >= 0.75:
    print('O modelo atingiu o limite mínimo de acurácia (0.75).')
else:
    print('O modelo NÃO atingiu o limite mínimo de acurácia (0.75).')

Acurácia no conjunto de teste: 0.7963
O modelo atingiu o limite mínimo de acurácia (0.75).


## 5. Tarefa adicional — prova real (sanity check)

Vamos comparar nosso modelo com um classificador "burro" (`DummyClassifier`), que sempre prevê a classe mais frequente. Se nosso modelo não superar esse baseline, algo está errado — ele não estaria aprendendo nada de útil.

In [25]:
dummy_model = DummyClassifier(strategy='most_frequent', random_state=12345)
dummy_model.fit(features_train, target_train)
dummy_predictions = dummy_model.predict(features_test)
dummy_accuracy = accuracy_score(target_test, dummy_predictions)

print(f'Acurácia do modelo burro (sempre prevê a classe majoritária): {dummy_accuracy:.4f}')
print(f'Acurácia do nosso modelo (floresta aleatória):                {test_accuracy:.4f}')

if test_accuracy > dummy_accuracy:
    print()
    print('Nosso modelo supera o baseline burro, então ele está de fato aprendendo padrões úteis nos dados.')
else:
    print()
    print('Nosso modelo NÃO supera o baseline burro — é preciso revisar o processo de treinamento.')

Acurácia do modelo burro (sempre prevê a classe majoritária): 0.6843
Acurácia do nosso modelo (floresta aleatória):                0.7963

Nosso modelo supera o baseline burro, então ele está de fato aprendendo padrões úteis nos dados.


## Conclusão geral

- Dividimos os dados em treinamento (60%), validação (20%) e teste (20%).
- Testamos três modelos de classificação (árvore de decisão, floresta aleatória e regressão logística), ajustando hiperparâmetros no conjunto de validação.
- A **floresta aleatória** apresentou a melhor acurácia na validação e foi escolhida como modelo final.
- No conjunto de teste, o modelo final atingiu uma acurácia acima do limite mínimo exigido de 0,75.
- O teste de sanidade confirma que o modelo está aprendendo padrões reais dos dados, superando um classificador ingênuo que sempre prevê a classe majoritária.